# Band Causal TCN: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.band_causal_tcn import BandCausalTCNModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import test_toolkit as ts
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "band_causal_tcn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El modelo *Band Causal TCN* se evaluó mediante un esquema de ventana expansiva con reentrenamiento periódico para el pronóstico de la volatilidad diaria del crudo Brent.
- En la comparación entre entrenamiento y prueba, QLIKE ($\Delta = -0.0637$) y HMSE ($\Delta = +1.2424$) registraron las mayores variaciones junto con cambios moderados en el resto de las métricas, presentando residuos con colas pesadas mejor descritos por una distribución $t$ de Student ($\nu = 5.9414$) que por la normal.
- En evaluación fuera de muestra, no se observó autocorrelación lineal, ni efectos ARCH significativos tras el ajuste de Bonferroni, ni dependencia no lineal residual.
- En el periodo de prueba y en el pico de volatilidad de 2026, el modelo siguió de forma consistente el nivel base en regímenes normales pero mostró limitaciones al predecir los picos durante los choques, capitalizando las arquitecturas convolucionales con margen de mejora en choques extremos y eficiencia computacional.
:::

## Introducción

El modelo *Band Causal TCN* (`BandCausalTCNModel`) es un modelo original de este proyecto que combina descomposición *wavelet* causal multiescala con convoluciones causales dilatadas residuales para pronosticar la volatilidad diaria del crudo Brent mediante la varianza realizada de Rogers-Satchell ($RS$). A partir de los resultados obtenidos en las arquitecturas evaluadas previamente, donde se observaron limitaciones de generalización ante cambios de régimen o calibración de picos extremos, este modelo adopta explícitamente una representación causal por bandas de frecuencia acoplada a una estructura convolucional causal.

La descomposición *wavelet* discreta causal permite separar la dinámica temporal en componentes de distinta escala de frecuencia sin utilizar información futura, mientras que los filtros causales por banda, la mezcla entre escalas y los bloques residuales dilatados capturan dependencias multiescala de corto y mediano plazo con estricta causalidad temporal.

El objetivo de esta sección es presentar, fundamentar y evaluar `BandCausalTCNModel` para el pronóstico de la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el mismo protocolo del *benchmark*. Se examina el impacto de la ventana histórica (`look_back`), la profundidad y estructura de la TCN (`num_levels`, `channels`, `kernel_size`, `num_convs`), la descomposición *wavelet* causal (`J`, `wavelet`, `decomp_window`, `band_kernel_size`) y la regularización (`dropout`, `weight_decay`), manteniendo la pérdida **QLIKE** y la evaluación *walk-forward* con reentrenamiento periódico.

### Arquitectura

La arquitectura de `BandCausalTCNNetwork` recibe una ventana de entrada de longitud $L$ (`look_back`) y calcula de forma estrictamente causal $B = J + 1$ bandas reconstruidas mediante transformada *wavelet* discreta (`pywt.wavedec` y `pywt.waverec` con modo simétrico). Para cada instante $t$ de la secuencia, la reconstrucción de cada banda $b \in \{1, \dots, J+1\}$ se evalúa únicamente sobre el prefijo causal acotado a una ventana de longitud efectiva `decomp_window` (con *padding* de borde si la ventana es menor que la longitud mínima requerida por la *wavelet* y el nivel $J$), extrayendo el valor final reconstruido de cada banda para conformar el tensor de bandas causales $\mathbf{X}_{\text{band}} \in \mathbb{R}^{B \times L}$.

Sobre cada banda $b$ se aplica una convolución causal unidimensional independiente (`CausalConv1d`, con *padding* exclusivamente a la izquierda) de tamaño de núcleo creciente por escala, $k_b = k_{\text{base}} + b - 1$ (donde $k_{\text{base}}$ es `band_kernel_size`), concatenando las respuestas en $\mathbf{H}_{\text{filt}} \in \mathbb{R}^{B \times L}$. A continuación, se combinan las representaciones filtradas mediante una convolución $1 \times 1$ con activación ReLU y se incorpora una conexión residual directa desde $\mathbf{X}_{\text{band}}$ mediante otra convolución $1 \times 1$ hacia el espacio de `channels` ($C$):

$$
\mathbf{H}_{\text{in}} = \text{ReLU}\left(\mathbf{W}_{\text{mix}} * \mathbf{H}_{\text{filt}} + \mathbf{b}_{\text{mix}}\right) + \left(\mathbf{W}_{\text{res}} * \mathbf{X}_{\text{band}} + \mathbf{b}_{\text{res}}\right),
$$

donde $\mathbf{H}_{\text{in}} \in \mathbb{R}^{C \times L}$ y $*$ denota convolución causal $1 \times 1$.

Posteriormente, $\mathbf{H}_{\text{in}}$ atraviesa una secuencia de `num_levels` bloques residuales (`TCNBlock`) con dilataciones exponenciales $d_l = 2^{l-1}$ para $l = 1, \dots, \text{num\_levels}$. En cada bloque, se encadenan `num_convs` capas de convolución causal dilatada (`CausalConv1d`), activación ReLU y *Dropout*, sumando una conexión residual (con proyección $1 \times 1$ si cambian los canales) seguida de una activación ReLU de salida:

$$
\mathbf{H}^{(l)} = \text{ReLU}\left(\mathcal{F}_l\left(\mathbf{H}^{(l-1)}\right) + \mathcal{R}_l\left(\mathbf{H}^{(l-1)}\right)\right),
$$

donde $\mathcal{F}_l$ representa la secuencia de convoluciones causales dilatadas con ReLU y *Dropout* del bloque $l$, y $\mathcal{R}_l$ es la rama residual correspondiente.

Finalmente, se toma el último paso temporal de la salida del bloque convolucional, $\mathbf{h}_L^{\text{num\_layers}} \in \mathbb{R}^{C}$, y se proyecta a una salida escalar estrictamente positiva mediante una capa lineal y la función *Softplus* con piso numérico $\varepsilon = 10^{-8}$:

$$
\widehat{RS}_t = \text{Softplus}\left(\mathbf{w}_{\text{fc}}^{\top}\mathbf{h}_L^{\text{num\_layers}} + b_{\text{fc}}\right) + \varepsilon = \ln\left(1 + e^{\mathbf{w}_{\text{fc}}^{\top}\mathbf{h}_L^{\text{num\_layers}} + b_{\text{fc}}}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ agrupa todos los parámetros entrenables de la red.

Esta formulación es distinta a una TCN estándar sobre la serie cruda e incluso a una TCN multi-entrada convencional con variables exógenas dadas: aquí las $J+1$ entradas se derivan de forma estrictamente causal desde la propia ventana mediante descomposición *wavelet* por escala, se filtran con núcleos causales adaptados a cada banda (`band_kernel_size + b - 1`) y se fusionan con un atajo residual directo antes de pasar por las capas dilatadas.

Siguiendo el protocolo del *benchmark*, el modelo se entrena y evalúa mediante *walk-forward* con reentrenamiento periódico (`retrain_freq=1`), estandarización Z-score ajustada en cada *fold* de entrenamiento para los retornos y escalamiento del objetivo $RS$ (`target_scale`), optimizando la pérdida asimétrica **QLIKE** en `PyTorch` (`2.13.0`) con diferenciación automática.

### Continuidad de Lipschitz y Estabilidad

En el pronóstico de volatilidad financiera, las series de retornos del crudo Brent presentan cambios abruptos de régimen, ruido de alta frecuencia y valores atípicos. Probar la continuidad de Lipschitz de `BandCausalTCNNetwork` es necesario para dar sustento formal a la estabilidad del mapeo entre la ventana de entrada y la varianza pronosticada, evitando que pequeñas variaciones o perturbaciones acotadas en los datos de entrada generen cambios desproporcionados o inestabilidades numéricas en $\widehat{RS}_t$.

Desde el punto de vista operativo y matemático, la continuidad de Lipschitz implica que la sensibilidad local y global de la red está controlada por una constante finita: si dos ventanas de entrada difieren en norma por una cantidad pequeña, la diferencia entre sus predicciones queda acotada proporcionalmente. Esto resulta especialmente relevante al minimizar una pérdida asimétrica como **QLIKE** en un esquema *walk-forward*, donde subestimaciones o sobreestimaciones bruscas ante picos o periodos de baja varianza pueden amplificar el error.

Para demostrar la continuidad de Lipschitz por composición de funciones, procedemos paso a paso. Si una función $f: \mathbb{R}^n \to \mathbb{R}^m$ satisface, para una norma fijada $\|\cdot\|$ y algún escalar $L_f \ge 0$:

$$
\|f(x_1) - f(x_2)\| \le L_f \|x_1 - x_2\| \quad \forall x_1, x_2 \in \mathbb{R}^n,
$$

y una segunda función $g: \mathbb{R}^m \to \mathbb{R}^p$ satisface $\|g(u_1) - g(u_2)\| \le L_g \|u_1 - u_2\|$ para todo $u_1, u_2 \in \mathbb{R}^m$, entonces por desigualdad triangular multiplicativa:

$$
\|(g \circ f)(x_1) - (g \circ f)(x_2)\| = \|g(f(x_1)) - g(f(x_2))\| \le L_g \|f(x_1) - f(x_2)\| \le L_g L_f \|x_1 - x_2\|.
$$

De este modo, descomponemos la red en cada bloque elemental, verificamos su constante de Lipschitz bajo la suposición de que los parámetros del modelo permanecen acotados (respaldada por la regularización `weight_decay`, la parada temprana y el control numérico durante el entrenamiento) y luego encadenamos las cotas a lo largo de todo el recorrido hacia adelante.

#### Paso 1: Extracción causal de bandas *wavelet*

Sea $\mathbf{x} \in \mathbb{R}^L$ la ventana de entrada de longitud $L$ (`look_back`). Para cada instante $t \in \{0, \dots, L-1\}$, el modelo toma el prefijo acotado $\mathbf{w}_t = \mathbf{x}_{s_t : t+1}$ con $s_t = \max(0, t + 1 - W_{\text{eff}})$ (donde $W_{\text{eff}}$ es `decomp_window`), aplica *padding* de borde (`mode="edge"`) si su longitud es menor que el mínimo exigido por la *wavelet* y el nivel $J$, y calcula la descomposición y reconstrucción por banda (`pywt.wavedec` y `pywt.waverec` con modo simétrico), quedándose con el último valor reconstruido de cada banda $b \in \{1, \dots, J+1\}$.

Tanto el recorte de ventana, el *padding* de borde, la transformada *wavelet* discreta como la reconstrucción lineal por banda son transformaciones lineales sobre las entradas de la ventana. Para una longitud de ventana fija, cualquier operador lineal $T_{\text{band}}: \mathbb{R}^L \to \mathbb{R}^{(J+1) \times L}$ entre espacios normados de dimensión finita es acotado, por lo que existe una constante finita:

$$
L_{\text{band}} = \|T_{\text{band}}\|_{\text{op}} < \infty,
$$

tal que, para dos ventanas de entrada $\mathbf{x}_1$ y $\mathbf{x}_2$:

$$
\|\mathbf{X}_{\text{band}}(\mathbf{x}_1) - \mathbf{X}_{\text{band}}(\mathbf{x}_2)\| \le L_{\text{band}} \|\mathbf{x}_1 - \mathbf{x}_2\|.
$$

#### Paso 2: Filtros causales por banda (`band_convs`)

Cada banda $b \in \{1, \dots, J+1\}$ pasa por una capa `CausalConv1d` unidimensional independiente con kernel de longitud $k_b = k_{\text{base}} + b - 1$ y dilatación $1$. El *padding* causal por la izquierda es un operador lineal con norma $1$ (solo añade ceros), y la convolución 1D es una aplicación lineal representada por una matriz de Toeplitz/banda de pesos acotados $\mathbf{W}_b$. Su constante de Lipschitz está acotada por su norma de operador:

$$
L_{\text{conv}, b} = \|\mathbf{W}_b\|_{\text{op}}.
$$

Al concatenar las $J+1$ bandas filtradas a lo largo del eje de canales para formar $\mathbf{H}_{\text{filt}} \in \mathbb{R}^{(J+1) \times L}$, la diferencia queda acotada por la constante conjunta $L_{\text{filt}}$ (determinada por los máximos o normas de bloque de $\{\mathbf{W}_b\}_{b=1}^{J+1}$ según la norma elegida):

$$
\|\mathbf{H}_{\text{filt}}(\mathbf{x}_1) - \mathbf{H}_{\text{filt}}(\mathbf{x}_2)\| \le L_{\text{filt}} \|\mathbf{X}_{\text{band}}(\mathbf{x}_1) - \mathbf{X}_{\text{band}}(\mathbf{x}_2)\|.
$$

#### Paso 3: Mezcla entre bandas y conexión residual (`band_mix` y `band_residual`)

A partir de $\mathbf{H}_{\text{filt}}$, la capa `band_mix` aplica una convolución $1 \times 1$ (proyección lineal canal a canal con pesos $\mathbf{W}_{\text{mix}}$ y sesgo $\mathbf{b}_{\text{mix}}$) seguida de la activación $\text{ReLU}(u) = \max(0, u)$. Dado que $|\text{ReLU}'(u)| \le 1$ en todo punto diferenciable, $\text{ReLU}$ es $1$-Lipschitz ($L_{\text{ReLU}} = 1$). Por composición:

$$
\|\text{ReLU}(\mathbf{W}_{\text{mix}} * \mathbf{H}_{\text{filt}}^{(1)} + \mathbf{b}_{\text{mix}}) - \text{ReLU}(\mathbf{W}_{\text{mix}} * \mathbf{H}_{\text{filt}}^{(2)} + \mathbf{b}_{\text{mix}})\| \le \|\mathbf{W}_{\text{mix}}\|_{\text{op}} \|\mathbf{H}_{\text{filt}}^{(1)} - \mathbf{H}_{\text{filt}}^{(2)}\|.
$$

En paralelo, la rama residual `band_residual` proyecta directamente $\mathbf{X}_{\text{band}}$ mediante una convolución $1 \times 1$ con pesos $\mathbf{W}_{\text{res}}$ y sesgo $\mathbf{b}_{\text{res}}$, cuya constante de Lipschitz es $\|\mathbf{W}_{\text{res}}\|_{\text{op}}$.

Como la suma de funciones Lipschitz-continuas es Lipschitz-continua con constante igual a la suma de sus constantes, el tensor de entrada a la TCN, $\mathbf{H}_{\text{in}} = \text{ReLU}(\mathbf{W}_{\text{mix}} * \mathbf{H}_{\text{filt}} + \mathbf{b}_{\text{mix}}) + (\mathbf{W}_{\text{res}} * \mathbf{X}_{\text{band}} + \mathbf{b}_{\text{res}})$, satisface:

$$
\|\mathbf{H}_{\text{in}}(\mathbf{x}_1) - \mathbf{H}_{\text{in}}(\mathbf{x}_2)\| \le \left(\|\mathbf{W}_{\text{mix}}\|_{\text{op}} L_{\text{filt}} L_{\text{band}} + \|\mathbf{W}_{\text{res}}\|_{\text{op}} L_{\text{band}}\right) \|\mathbf{x}_1 - \mathbf{x}_2\| = L_{\text{in}} \|\mathbf{x}_1 - \mathbf{x}_2\|.
$$

#### Paso 4: Bloques residuales TCN dilatados (`TCNBlock`)

Cada bloque $l \in \{1, \dots, \text{num\_levels}\}$ contiene `num_convs` capas `CausalConv1d` con dilatación $d_l = 2^{l-1}$, intercaladas con `ReLU` ($1$-Lipschitz) y `Dropout` con probabilidad $p \in [0, 1)$. En modo de inferencia, `Dropout` es la identidad ($L_{\text{drop}} = 1$); en modo de entrenamiento, el escalamiento por $1/(1-p)$ es lineal con constante $L_{\text{drop}} = (1-p)^{-1}$ para $p < 1$.

Si denotamos por $\|\mathbf{W}_{l,c}\|_{\text{op}}$ la norma de operador de la $c$-ésima convolución dilatada del bloque $l$, la rama convolucional secuencial $\mathcal{F}_l$ tiene constante acotada por:

$$
L_{\mathcal{F}_l} \le \prod_{c=1}^{\text{num\_convs}} \left(\|\mathbf{W}_{l,c}\|_{\text{op}} \cdot L_{\text{drop}}\right).
$$

La conexión residual $\mathcal{R}_l$ es la identidad (constante $1$) si los canales no cambian, o una convolución $1 \times 1$ con constante $\|\mathbf{W}_{\text{down}, l}\|_{\text{op}}$ si hay proyección de canales. Al sumar ambas ramas y aplicar la activación final $\text{ReLU}$ ($1$-Lipschitz), la constante del bloque $l$ queda acotada por:

$$
L_{\text{block}, l} \le 1 \cdot \left(L_{\mathcal{F}_l} + L_{\mathcal{R}_l}\right) = L_{\mathcal{F}_l} + L_{\mathcal{R}_l}.
$$

Encadenando los `num_levels` bloques de la TCN, el bloque convolucional completo tiene constante de Lipschitz:

$$
L_{\text{TCN}} \le \prod_{l=1}^{\text{num\_levels}} L_{\text{block}, l}.
$$

#### Paso 5: Extracción del último paso, capa lineal y salida *Softplus*

Tomar el último paso temporal de la secuencia es una proyección lineal de selección de coordenadas con constante $L_{\text{last}} = 1$. A continuación, la capa densa `fc` aplica la transformación lineal $\mathbf{w}_{\text{fc}}^{\top}\mathbf{h} + b_{\text{fc}}$, cuya constante de Lipschitz es:

$$
L_{\text{fc}} = \|\mathbf{w}_{\text{fc}}\|.
$$

Finalmente, la función $\text{Softplus}(u) = \ln(1 + e^u)$ tiene primera derivada:

$$
\frac{d}{du}\text{Softplus}(u) = \frac{e^u}{1 + e^u} = \sigma(u) \in (0, 1) \quad \forall u \in \mathbb{R}.
$$

Por el teorema del valor medio, $|\text{Softplus}(u_1) - \text{Softplus}(u_2)| \le 1 \cdot |u_1 - u_2|$, por lo que $\text{Softplus}$ es $1$-Lipschitz ($L_{\text{softplus}} = 1$), y sumar la constante $\varepsilon = 10^{-8}$ no altera la diferencia.

#### Cota global de Lipschitz

Componiendo todas las etapas desde la ventana de entrada hasta la varianza pronosticada $\widehat{RS}_t$, obtenemos una constante global finita:

$$
L_{\text{total}} \le L_{\text{band}} \cdot \left(\|\mathbf{W}_{\text{mix}}\|_{\text{op}} L_{\text{filt}} + \|\mathbf{W}_{\text{res}}\|_{\text{op}}\right) \cdot L_{\text{TCN}} \cdot L_{\text{fc}} \cdot L_{\text{softplus}}.
$$

Con esto queda establecido que, bajo pesos acotados, el mapeo de `BandCausalTCNNetwork` es Lipschitz-continuo y no amplifica sin cota perturbaciones acotadas en la serie de retornos.

## Entrenamiento

Para la selección de hiperparámetros del modelo `BandCausalTCNModel`, se mantuvo el mismo protocolo experimental (*Walk-Forward*) y el mismo presupuesto computacional que en los demás modelos del *benchmark*. A diferencia de las arquitecturas sin descomposición espectral previa, la búsqueda con Optuna exploró conjuntamente la estructura multiescala de las bandas *wavelet* causales, el bloque convolucional dilatado y los parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema *Walk-Forward* (`n_splits=5`, `embargo=1`, y `warmup` definido según la ejecución, con `exec_id = 2` y `current_warmup = 1220`). Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, utilizada como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque preservó el orden temporal de las observaciones y evitó cualquier fuga de información futura (*look-ahead bias*) durante el entrenamiento y el reentrenamiento periódico (`retrain_freq=1`).

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el tamaño del núcleo convolucional (`kernel_size`), el número de convoluciones por bloque (`num_convs`) y el número de niveles dilatados (`num_levels`), acotado entre un mínimo calculado según la cobertura temporal requerida y un máximo superior. Asimismo, se exploraron el ancho de canales (`channels`), la tasa de abandono (`dropout`), la tasa de aprendizaje (`learning_rate`), la penalización $\ell_2$ (`weight_decay`), el nivel de descomposición *wavelet* (`J`), la familia *wavelet* (`wavelet`) y el tamaño base del filtro por banda (`band_kernel_size`), fijando `seed = 42`.

El entrenamiento de cada red se configuró con un máximo de 200 épocas (`max_epochs = 200`) y una paciencia de 20 épocas (`patience = 20`) para la parada temprana interna, bajo un esquema de búsqueda con `n_trials = 50` y `early_stop_rounds = 15`. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def band_causal_tcn_search_space(trial: optuna.Trial) -> dict:
    look_back = trial.suggest_categorical("look_back", [7, 14, 21, 28])
    kernel_size = trial.suggest_categorical("kernel_size", [2, 3])
    num_convs = trial.suggest_categorical("num_convs", [1, 2, 3])

    min_levels = max(
        1,
        math.ceil(
            math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
        ),
    )
    max_levels = min(min_levels + 2, 5)

    return {
        "look_back": look_back,
        "kernel_size": kernel_size,
        "num_convs": num_convs,
        "num_levels": trial.suggest_int("num_levels", min_levels, max_levels),
        "channels": trial.suggest_categorical("channels", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "J": trial.suggest_categorical("J", [1, 2, 3]),
        "wavelet": trial.suggest_categorical("wavelet", ["db4", "sym4"]),
        "band_kernel_size": trial.suggest_categorical("band_kernel_size", [1, 2]),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 2
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=BandCausalTCNModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=band_causal_tcn_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

Aunque el protocolo experimental contemplaba un presupuesto de 50 ensayos (`n_trials=50`), el estudio de Optuna registró 16 ensayos en esta ejecución, completando 16 ensayos válidos (100.0 %), con 0 ensayos podados y 0 fallidos, y alcanzando un mejor valor de QLIKE de 0.4626. La ausencia de podas refleja que ninguna configuración fue descartada prematuramente durante la búsqueda, mientras que la arquitectura con descomposición *wavelet* causal y bloques convolucionales dilatados estabilizó su entrenamiento mediante conexiones residuales y regularización para completar todas las evaluaciones registradas.

La búsqueda en Optuna tomó aproximadamente 10 horas y 44 minutos, mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó 352.60 segundos (unos 5.88 minutos), con un promedio de 4.4080 segundos por partición. Este tiempo de ajuste evidencia el costo computacional de entrenar redes convolucionales causales con extracción *wavelet* por ventana frente a modelos más ligeros, conservando la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [2]:
resumen_data = []

exec_id = 1
model_tag = "BandCausalTCN"
study_name_override = None
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = sorted(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    if "train_time_sec" in df_csv.columns:
        tiempo_total_rep = float(df_csv["train_time_sec"].sum())
        tiempo_promedio_fold = float(df_csv["train_time_sec"].mean())
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        available_studies = optuna.get_all_study_names(storage=storage_url)

        if study_name_override and study_name_override in available_studies:
            target_study = study_name_override
        elif available_studies:
            target_study = available_studies[-1]
        else:
            target_study = f"{model_tag}_study"

        study = optuna.load_study(study_name=target_study, storage=storage_url)

        total_trials = len(study.trials)
        completos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        fallidos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)

        if completos > 0 and study.best_value is not None:
            mejor_qlike = round(float(study.best_value), 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Exec": exec_id,
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Exec,Total Trials,Completos,Podados,Fallidos,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
1,16,16,0,0,0.4626,352.6000,4.4080


El ranking de las cinco mejores configuraciones mostró un consenso unánime en el uso de ventanas cortas de 7 días (`Lookback = 7`) y en la *wavelet* `db4` en todas las posiciones, lo que sugiere que el historial reciente de una semana y la descomposición *wavelet* de corto y mediano alcance resultan informativos para capturar la dinámica de volatilidad del Brent. En cuanto a la estructura multiescala y convolucional, las cinco configuraciones emplearon `Band K` = 1, 3 convoluciones por bloque (`Convs` = 1), niveles de descomposición `J` y bloques TCN dilatados niveles con 64 o 128 canales. Cuando Band K = 1, el filtro causal inicial sobre la primera banda opera punto a punto en el tiempo y los kernels crecen progresivamente en las bandas de mayor índice, mientras que la memoria histórica se apoya en las bandas *wavelet* causales y en las capas dilatadas posteriores. A diferencia de las arquitecturas recurrentes, el acoplamiento entre bandas causales y convoluciones causales dilatadas permite capturar relaciones multiescala sobre la ventana sin recurrencia paso a paso.

In [3]:
csv_path = exec_dir / "BandCausalTCN_results.csv"
if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("*_results.csv"))
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_bct = pd.read_csv(csv_path)

df_bct["lr"] = df_bct["learning_rate"].apply(lambda x: f"{x:.4f}")
df_bct["wd"] = df_bct["weight_decay"].apply(lambda x: f"{x:.1e}")
df_bct["drop_fmt"] = df_bct["dropout"].apply(lambda x: f"{x:.3f}")

cols_agrupar = [
    "look_back",
    "J",
    "wavelet",
    "band_kernel_size",
    "num_levels",
    "channels",
    "kernel_size",
    "num_convs",
    "drop_fmt",
    "lr",
    "wd",
]

df_agg = (
    df_bct.groupby(cols_agrupar, as_index=False)["qlike"]
    .agg(mean="mean", std="std")
)
df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

df_top5 = df_agg.sort_values(by="mean", ascending=True).head(5).rename(
    columns={
        "look_back": "Lookback",
        "J": "Nivel J",
        "wavelet": "Wavelet",
        "band_kernel_size": "Band K",
        "num_levels": "Niveles TCN",
        "channels": "Canales",
        "kernel_size": "Kernel",
        "num_convs": "Convs",
        "drop_fmt": "Dropout",
        "lr": "LR",
        "wd": "Weight Decay",
    }
)

cols_finales = [
    "Lookback",
    "Nivel J",
    "Wavelet",
    "Band K",
    "Niveles TCN",
    "Canales",
    "Kernel",
    "Convs",
    "Dropout",
    "LR",
    "Weight Decay",
    "QLIKE",
]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Nivel J,Wavelet,Band K,Niveles TCN,Canales,Kernel,Convs,Dropout,LR,Weight Decay,QLIKE
7,2,db4,1,4,128,2,3,0.248,0.0010,9.2e-04,0.463 ± 0.154
7,1,db4,1,3,128,2,3,0.192,0.0007,7.1e-04,0.522 ± 0.216
7,1,db4,1,3,64,2,3,0.177,0.0004,6.4e-04,0.535 ± 0.276
7,3,db4,1,1,128,3,3,0.156,0.0011,7.0e-04,0.545 ± 0.256
7,2,db4,1,3,128,2,3,0.212,0.0011,9.8e-05,0.595 ± 0.319


Entre las dos primeras configuraciones, el primer lugar obtuvo una menor pérdida media de validación (0.463 frente a 0.522) y una menor dispersión entre pliegues, pese a emplear un mayor ancho de canales, mayor profundidad TCN y nivel *wavelet* 2 frente a 1. La elección de la primera configuración responde a su menor QLIKE medio de validación y a su mayor estabilidad relativa entre *folds* dentro del rango de las mejores soluciones, manteniendo un desempeño superior al de las configuraciones restantes, cuya dispersión aumenta a partir del tercer lugar.

In [ ]:
best_band_tcn_model = BandCausalTCNModel(
    look_back=7,
    num_levels=4,
    channels=128,
    kernel_size=2,
    num_convs=3,
    dropout=0.248,
    learning_rate=0.0010,
    weight_decay=9.2e-04,
    J=2,
    wavelet="db4",
    band_kernel_size=1,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_band_tcn_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time

El entrenamiento del modelo sobre toda la muestra tuvo una duración de 15.96 segundos (15.9568 s). En la gráfica de aprendizaje, tras un valor inicial elevado de entrenamiento que supera el límite superior del eje y un descenso rápido en validación en la segunda época, ambas trayectorias reducen su pérdida QLIKE en las primeras épocas hasta estabilizarse en un rango bajo. Los mejores pesos se alcanzan en la época 12, y el entrenamiento continúa hasta la época 32 bajo el criterio de parada temprana.

A diferencia de *Causal Transformer*, donde la pérdida de validación superó con rapidez y de forma persistente a la de entrenamiento tras la selección de mejores pesos, aquí ambas curvas se mantienen cercanas y estables, con la validación ligeramente por debajo o a la par del entrenamiento y sin divergencias. La meseta posterior indica que el modelo conserva una convergencia estable y una generalización controlada sobre la varianza condicional, con ajustes moderados en los picos extremos.

In [9]:
import pandas as pd

history_csv_path = exec_dir / "band_causal_tcn_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

n_epochs = len(history_df)
x_max = max(n_epochs + 1, 2)
x_step = max(1, round(x_max / 11))

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14,
    line_width=2.0, title_size=14, label_size=14,
    tick_size=12, legend_size=14, ylim=(0, 6),
    xlim=(0, x_max), y_step=0.5, x_step=x_step,
    legend_bbox=(0.95, 0.95),
)

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no fue posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) sobre la serie completa de entrenamiento: primero se transformó `y_train` con el `StandardScaler` ajustado durante `fit` y se escaló el objetivo de varianza, luego se construyó la matriz de rezagos con `TimeSeriesDataset` usando ventanas de 7 días (`look_back = 7`) y finalmente se evaluó en modo `eval()` sin gradientes ni regularización. Las varianzas estimadas se reescalaron a su unidad original y se alinearon con los datos reales descartando las observaciones iniciales correspondientes a la ventana.

In [ ]:
y_train_scaled = best_band_tcn_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_eff = rs_train * best_band_tcn_model.target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_eff,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=best_band_tcn_model.batch_size,
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_band_tcn_model.device)
        preds_batch = best_band_tcn_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_band_tcn_model.target_scale

y_train_aligned = y_train[look_back:]
rs_train_aligned = rs_train[look_back:]

df_train_preds = pd.DataFrame({
    "y_true": y_train_aligned,
    "rs_true": rs_train_aligned,
    "y_pred": preds_train_var,
})

train_output_path = exec_dir / "band_causal_tcn_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=False)

A partir de las varianzas condicionales predichas $\left(\hat{\sigma}_t^2\right)$ y de los retornos observados $\left(y_t\right)$, se evaluó el ajuste del modelo sobre el conjunto de entrenamiento mediante los residuos estandarizados:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El diagnóstico sobre los residuos estandarizados mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .326$), así como ausencia de efectos ARCH en los mismos rezagos ($p_{\text{adj}} \ge .245$, sin evidencia de agrupamiento residual de volatilidad) y ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .202$). Por su parte, las pruebas de Jarque-Bera y Lilliefors rechazaron la hipótesis de normalidad de los residuos ($p < .001$), lo que refleja colas pesadas y asimetría características de los retornos del crudo Brent.

In [10]:
import numpy as np
import pandas as pd

train_output_path = exec_dir / "band_causal_tcn_train_predictions.csv"

df_train_preds = pd.read_csv(train_output_path)
rs_train_aligned = df_train_preds["rs_true"].to_numpy()
preds_train_var = df_train_preds["y_pred"].to_numpy()

n_preds = len(preds_train_var)
y_arr = np.asarray(y_train).reshape(-1)
y_train_aligned = y_arr[-n_preds:]

std_residuals_train = y_train_aligned / np.sqrt(np.maximum(preds_train_var, 1e-12))
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),7.0069,.725,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),28.0375,.109,.326,Sin Autocorrelación
Ljung-Box (Lag 30),39.3037,.119,.357,Sin Autocorrelación
ARCH-LM (Lag 10),15.8273,.105,.314,Sin Efectos ARCH
ARCH-LM (Lag 20),29.3150,.082,.245,Sin Efectos ARCH
ARCH-LM (Lag 30),36.1257,.204,.612,Sin Efectos ARCH
Jarque-Bera,1295.3913,< .001,--,Residuos No Normales
Lilliefors,0.0593,< .001,--,Residuos No Normales
BDS (Dim 2),1.2754,.202,--,Sin Estructura No Lineal


Los residuos estandarizados del modelo mostraron un ajuste a una distribución $t$ de Student ($\nu = 4.3127$). Tanto en la densidad empírica como en el gráfico Q-Q, el núcleo central y la mayor parte de la distribución siguen de cerca la curva teórica, mientras que en las colas extremas se observan ligeras desviaciones con los cuantiles superiores empíricos situándose por debajo de la línea de referencia. Estas discrepancias implican que la aproximación $t$ de Student sobrestima ligeramente la magnitud de los eventos extremos frente a la cola empírica de los residuos, reflejando un ajuste más preciso en la región central que en los valores atípicos más severos. Pese a ello, las pruebas de bondad de ajuste (Kolmogorov-Smirnov: $p = .6152$; Cramér-von Mises: $p = .3507$) no rechazaron la hipótesis nula de ajuste a la distribución $t$.

In [12]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.8),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal (estadístico $= 229.76$, $p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 227.76$ y $\Delta\text{BIC} = 222.08$ a favor de la distribución $t$ de Student. Estos resultados apoyan que la distribución $t$ captura con mayor precisión las colas pesadas de los residuos estandarizados del modelo en la muestra de entrenamiento en comparación con el ajuste Gaussiano.

In [14]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-2,916.34","5,836.68","5,848.04",-,-
t-Student,"-2,801.46","5,608.92","5,625.96",229.76,< .001


La inspección visual de las predicciones en el conjunto de entrenamiento muestra que *Band Causal TCN* sigue el nivel general de la volatilidad realizada ($\sigma_R$) en los periodos de calma relativa y responde con rapidez ante los episodios de tensión, incluyendo el choque de principios de 2020 y el repunte de 2022. En los tramos de baja volatilidad, la trayectoria pronosticada ($\hat{\sigma}_t$) se mantiene suave y ligeramente por encima o en torno a la volatilidad realizada, mientras que en el recuadro de la crisis del COVID-19 capta el ascenso y la disipación de los picos principales sin un ajuste tan marcado sobre los extremos aislados como el observado en *Causal Transformer*. Este comportamiento refleja un seguimiento equilibrado de los cambios de régimen en la muestra de entrenamiento, moderando las oscilaciones bruscas sin perder reacción frente a los choques de mercado.

In [16]:
look_back = 7
n_preds = len(preds_train_var)
start_idx = len(y_train) - n_preds

dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(
    np.maximum(np.asarray(rs_train)[start_idx : start_idx + n_preds], 0.0)
)
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Band Causal TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Band Causal TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su ajuste sobre los datos observados con su capacidad predictiva en observaciones no vistas. Mientras que el conjunto de entrenamiento permite verificar la convergencia del algoritmo sobre la dinámica histórica, la evaluación fuera de muestra determina el desempeño del modelo ante nuevos regímenes de mercado.

Para diagnosticar posibles problemas de generalización, se aplicaron los criterios pre-registrados del protocolo experimental a partir de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{\text{abs}} = M_{\text{test}} - M_{\text{train}}
$$

Se fijó un umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ para las métricas evaluadas, de modo que discrepancias por encima de este margen señalan cambios relevantes entre el ajuste en muestra y el desempeño fuera de muestra.

En el modelado de volatilidad con pérdida QLIKE, esta lectura debe matizarse por la asimetría de la función de pérdida, que castiga con mayor severidad la subestimación de la varianza que su sobreestimación. Un modelo con sesgo alcista o defensivo puede registrar un QLIKE menor en prueba que en entrenamiento sin que ello implique una mejor capacidad estructural, sino una coincidencia favorable entre el nivel predicho y el régimen de volatilidad del periodo de prueba. Por ello, el análisis de $\Delta_{\text{abs}}$ se complementa con la inspección de las trayectorias predichas y el diagnóstico de residuos estandarizados.

Las predicciones fuera de muestra se generaron mediante el método `forecast` del modelo, aplicando una ventana expansiva con reentrenamiento periódico (`retrain_freq = 1`) sin fuga de información y devolviendo las varianzas condicionales en su escala original para compararlas con la volatilidad realizada.

In [ ]:
start_time = time.time()
preds_test = best_band_tcn_model.forecast(y_seq=y_test, rs_seq=rs_test)
preds_time = time.time() - start_time

print(f"Tiempo de pronóstico: {preds_time / 60:.2f} minutos")

df_preds = pd.DataFrame({
    "date": df_test.index,
    "rs_true": rs_test,
    "y_pred": preds_test,
})

output_path = exec_dir / "band_causal_tcn_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 149.31 minutos (unos 8,958.60 s) para las 953 observaciones del conjunto de prueba, lo que equivale a un tiempo promedio 9.40 segundos por observación.

El diagnóstico de los residuos estandarizados fuera de muestra mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .062$), ausencia de efectos ARCH en los mismos rezagos ($p_{\text{adj}} \ge .379$, sin evidencia de agrupamiento residual de volatilidad) y ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .294$). Por su parte, las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$), lo que refleja colas pesadas en los retornos fuera de muestra del crudo Brent.

In [17]:
import numpy as np
import pandas as pd

preds_path = exec_dir / "band_causal_tcn_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values

std_residuals_oos = y_test / np.sqrt(np.maximum(preds_test_var, 1e-12))
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),11.0762,.352,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),34.9244,.021,.062,Sin Autocorrelación
Ljung-Box (Lag 30),41.0746,.086,.257,Sin Autocorrelación
ARCH-LM (Lag 10),15.1632,.126,.379,Sin Efectos ARCH
ARCH-LM (Lag 20),25.6505,.178,.533,Sin Efectos ARCH
ARCH-LM (Lag 30),35.3707,.229,.688,Sin Efectos ARCH
Jarque-Bera,165.3132,< .001,--,Residuos No Normales
Lilliefors,0.0415,< .001,--,Residuos No Normales
BDS (Dim 2),-1.0488,.294,--,Sin Estructura No Lineal


La inspección visual de los residuos estandarizados fuera de muestra muestra un ajuste general adecuado a la distribución $t$ de Student con $\nu = 5.9414$ grados de libertad. Los cuantiles muestrales siguen de cerca la diagonal teórica a lo largo del centro y gran parte de los extremos, observándose únicamente ligeras desviaciones por debajo de la línea de referencia en torno a los cuantiles inferiores (alrededor de $-4$ a $-2$) y en el extremo superior, junto con un valor aislado por encima de la diagonal cerca de $4.2$. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .8264$; Cramér-von Mises: $p = .7888$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [18]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal fuera de muestra ($p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 51.70$ y $\Delta\text{BIC} = 46.84$ a favor de la distribución $t$ de Student (log-verosimilitud de $-1,389.77$ frente a $-1,416.62$). Estos resultados apoyan que la distribución $t$ captura con mayor precisión las colas pesadas de los residuos estandarizados del modelo en el conjunto de prueba en comparación con el ajuste Gaussiano.

In [19]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,416.62","2,837.24","2,846.96",-,-
t-Student,"-1,389.77","2,785.54","2,800.12",53.69,< .001


La comparación entre entrenamiento y prueba muestra que el QLIKE mejora fuera de muestra ($\Delta = -0.0637$), junto con ligeros descensos en CRPS ($\Delta = -0.0003$) y en el cuantil inferior de Pinball, mientras que métricas sensibles a picos relativos como HMSE y HMAE aumentan y las pérdidas cuadráticas y absolutas registran incrementos marginales o nulos ($\Delta \text{RMSE} = +0.0006$, $\Delta \text{MAE} = -0.0000$). Al igual que en *Causal Transformer* y la TCN, la reducción del QLIKE promedio en prueba frente al entrenamiento responde al cambio de régimen de volatilidad y a la ventana de evaluación de prueba, donde los episodios de mayor dispersión relativa elevan HMSE pese a contenerse la pérdida asimétrica principal.

In [20]:
look_back = 7
rs_train_aligned = np.asarray(rs_train)[look_back:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia",
    ),
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4659,0.4022,-0.0637
CRPS,0.0127,0.0124,-0.0003
Pinball (τ=0.05),0.0031,0.0027,-0.0003
Pinball (τ=0.95),0.0014,0.0016,+0.0002
MSE,0.0000,0.0000,+0.0000
RMSE,0.0017,0.0023,+0.0006
MAE,0.0005,0.0005,-0.0000
HMSE,0.8476,2.0900,+1.2424
HMAE,0.6305,0.6927,+0.0623


En el ajuste fuera de muestra, *Band Causal TCN* sigue en términos generales la trayectoria de la volatilidad realizada $\left(\sigma_R\right)$ a lo largo de la ventana de prueba, reproduciendo de forma adecuada el nivel base durante los periodos de régimen normal de forma coherente con las arquitecturas evaluadas previamente. En el recuadro correspondiente al pico de volatilidad de 2026 se observa que el modelo reacciona ante el incremento de la dispersión, aunque aún existe margen para refinar la captura del pico máximo y el ajuste fino en los episodios de mayor tensión. Aun así, la serie pronosticada ($\hat{\sigma}_t$) acompaña la evolución de la volatilidad realizada de manera consistente tanto en las fases tranquilas como en la disipación posterior a los choques.

In [21]:
import numpy as np
import pandas as pd

df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(np.maximum(df_preds_test["rs_true"], 0.0))
pred_test_vol = np.sqrt(np.maximum(df_preds_test["y_pred"], 0.0))

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste Fuera de la Muestra (Band Causal TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Band Causal TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La implementación y evaluación del modelo *Band Causal TCN* permitieron contrastar el desempeño de una arquitectura con descomposición *wavelet* causal y bloques convolucionales dilatados para el pronóstico de la volatilidad diaria del crudo Brent. En la comparación entre entrenamiento y prueba, el modelo varió en QLIKE ($\Delta = -0.0637$) y en HMSE ($\Delta = +1.2424$), dado que estas diferencias se asocian a cambios de régimen de volatilidad y a la transición hacia la evaluación secuencial fuera de muestra, mientras que el resto de las métricas registraron cambios reducidos.

En el diagnóstico fuera de muestra, los residuos estandarizados no mostraron autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30, ni efectos ARCH significativos en dichos rezagos, ni dependencia no lineal en la prueba BDS en dimensión 2. Asimismo, se rechazó la normalidad en Jarque-Bera y Lilliefors, y los residuos presentaron colas pesadas mejor representadas por una distribución $t$ de Student ($\nu = 5.9414$) que por la normal, con pruebas de Kolmogorov-Smirnov y Cramér-von Mises y de razón de verosimilitud favorables a este ajuste.

En términos de trayectoria y cómputo, el modelo siguió de forma consistente la volatilidad realizada en los regímenes habituales, pero mostró limitaciones al predecir los picos durante los choques del conjunto de prueba (como el episodio de 2026), y el esquema secuencial con reentrenamiento periódico requirió 149.31 minutos (unos 8,958.60 s) para las 953 observaciones de prueba (0.16 minutos o unos 9.40 segundos por observación en promedio). En conjunto, *Band Causal TCN* capitaliza el desempeño de las arquitecturas convolucionales y ofrece una estimación estable de la volatilidad, pero mantiene margen de mejora tanto en la captura de choques extremos fuera de muestra como en la eficiencia computacional del reentrenamiento.